# WAFER-GUARD · Fab-7 Excursion Detector (repaired)

This is the audited and repaired version of the original notebook. Every fix is marked **`# FIX`**, and the evidence for each defect is reproduced in **Section A** before the repaired pipeline runs.

**Summary of repairs**
1. **Data leakage.** The original fit the imputer, scaler and SMOTE on the whole dataset, then made a random split. SMOTE ran before the split, so synthetic copies of training defects landed in the test set, and the test set was 50 % defects. Now the split is chronological (train → val → test), every preprocessing step is fit on the training window only, and class balance is handled inside training.
2. **Corrupted reconstruction gradients.** The input was `[B,1,F]` but the reconstruction was `[B,F]`, so `reconstructed - batch_x` silently broadcast to `[B,B,F]`. Now the input is `[B,F]` and every loss term is guarded by shape asserts.
3. **Invariant blindness (legacy probe station PS-L).** `sensor_0 == 0.0` is a sentinel, and `sensor_1` holds firmware codes {100, 200, 300} for those wafers. Now there is an explicit `is_legacy` flag and one-hot code columns, and both sensors are masked out of continuous scaling and imputation for legacy rows.
4. **Evaluation.** The model is evaluated once on the untouched, most-recent 20 % of lots at natural prevalence. The decision threshold is chosen on the validation window, never on test.

In [1]:
# Uncomment on Colab
# !pip install "torch>=2.2" "scikit-learn>=1.2" "imbalanced-learn>=0.11" pandas numpy joblib matplotlib

In [2]:
import os, random, copy, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                             average_precision_score, confusion_matrix, precision_recall_curve)
from imblearn.over_sampling import SMOTE
import joblib

warnings.filterwarnings("ignore")
SEED = 1337
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HP = dict(batch_size=64, epochs=60, lr=1e-3, weight_decay=1e-4, hidden=256, latent=32,
          lambda_clf=1.0, patience=10, clip=10.0)
print(f"torch {torch.__version__} | device={DEVICE}")

torch 2.14.1+cu130 | device=cpu


## 1 · Data

In [3]:
df = pd.read_csv("secom_telemetry_modified.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)   # FIX: enforce time order
SENSOR_COLS = [c for c in df.columns if c.startswith("sensor_")]
y_all = (df["label"].to_numpy() == 1).astype(np.int64)
print(f"rows {len(df)} | sensors {len(SENSOR_COLS)} | defect rate {y_all.mean():.2%} | "
      f"{df.timestamp.min()} → {df.timestamp.max()}")

rows 1567 | sensors 590 | defect rate 6.64% | 2008-07-19 11:55:00 → 2008-10-17 06:07:00


## A · Audit evidence (re-running the original pipeline's logic)

Each cell below reproduces one defect with numbers. None of these cells feed the repaired model.

### A1 · Leakage: SMOTE before the split + global preprocessing + random split

In [4]:
X_raw = df[SENSOR_COLS].to_numpy(np.float64)
Xs_orig = StandardScaler().fit_transform(SimpleImputer(strategy="mean", keep_empty_features=True).fit_transform(X_raw))
Xb, yb = SMOTE(random_state=SEED).fit_resample(Xs_orig, y_all)
idx = np.arange(len(yb))
Xtr_o, Xte_o, ytr_o, yte_o, itr_o, ite_o = train_test_split(Xb, yb, idx, test_size=0.2, stratify=yb, random_state=SEED)
syn = ite_o >= len(y_all)                     # SMOTE appends synthetic rows after the real ones
print(f"original test prevalence: {yte_o.mean():.1%}  (natural is {y_all.mean():.1%})")
print(f"synthetic rows in original test set: {syn.sum()} of {yte_o.sum()} 'defective' test rows")
nn_ = NearestNeighbors(n_neighbors=1).fit(Xtr_o)
d_syn  = nn_.kneighbors(Xte_o[syn])[0].ravel()
d_real = nn_.kneighbors(Xte_o[(~syn) & (yte_o == 0)])[0].ravel()
print(f"median distance to nearest TRAIN row: synthetic test defects {np.median(d_syn):.1f} "
      f"vs real test passes {np.median(d_real):.1f}  -> test defects are near-copies of training data")
ts = df.timestamp.to_numpy()
_, _, _, _, a, b = train_test_split(X_raw, y_all, np.arange(len(y_all)), test_size=0.2, random_state=SEED)
print(f"random split: {(ts[b] < ts[a].max()).mean():.1%} of test lots are OLDER than the newest training lot (future → past leakage)")

original test prevalence: 50.0%  (natural is 6.6%)
synthetic rows in original test set: 274 of 293 'defective' test rows
median distance to nearest TRAIN row: synthetic test defects 2.4 vs real test passes 17.8  -> test defects are near-copies of training data
random split: 99.7% of test lots are OLDER than the newest training lot (future → past leakage)


### A2 · Tensor broadcasting in the reconstruction loss

In [5]:
xb_demo  = torch.randn(64, 1, 590)          # what make_loader produced (unsqueeze(1))
rec_demo = torch.randn(64, 590)             # what the decoder returns after flatten()
print("reconstructed - batch_x ->", tuple((rec_demo - xb_demo).shape), " expected (64, 590)")
# Consequence: each reconstruction is scored against EVERY wafer in the batch, so the optimum is the
# batch mean, not the input. Proof: the best possible 'broadcast' loss equals the batch variance.
x2 = xb_demo.squeeze(1)
mean_pred = x2.mean(0, keepdim=True).expand_as(x2)
print(f"broadcast loss if decoder outputs the batch mean: {torch.mean((mean_pred - xb_demo)**2):.3f}")
print(f"broadcast loss if decoder outputs a PERFECT reconstruction: {torch.mean((x2 - xb_demo)**2):.3f}  (worse!)")

reconstructed - batch_x -> (64, 64, 590)  expected (64, 590)
broadcast loss if decoder outputs the batch mean: 0.980
broadcast loss if decoder outputs a PERFECT reconstruction: 1.961  (worse!)


### A3 · Legacy probe station PS-L (sentinel + firmware codes)

In [6]:
leg = (df["sensor_0"] == 0.0).to_numpy()
print(f"legacy wafers (sensor_0 == 0.0): {leg.sum()} ({leg.mean():.1%})")
print("sensor_1 on legacy wafers:", df.loc[leg, "sensor_1"].value_counts().to_dict())
print(f"sensor_1 on normal wafers: {df.loc[~leg,'sensor_1'].min():.0f} – {df.loc[~leg,'sensor_1'].max():.0f} (continuous)")
imp = SimpleImputer(strategy="mean").fit(X_raw); sc = StandardScaler().fit(imp.transform(X_raw))
print(f"original imputer mean  sensor_0: {imp.statistics_[0]:.0f}  (true non-legacy {np.nanmean(X_raw[~leg,0]):.0f})")
print(f"original imputer mean  sensor_1: {imp.statistics_[1]:.0f}  (true non-legacy {np.nanmean(X_raw[~leg,1]):.0f})")
print(f"original scaler std    sensor_1: {np.sqrt(sc.var_[1]):.0f}  (true non-legacy {np.nanstd(X_raw[~leg,1]):.0f})"
      "  -> genuine variation squashed")
print(f"defect rate: legacy {y_all[leg].mean():.1%} vs normal {y_all[~leg].mean():.1%}  -> the flag itself carries signal")

legacy wafers (sensor_0 == 0.0): 146 (9.3%)
sensor_1 on legacy wafers: {100.0: 61, 300.0: 50, 200.0: 35}
sensor_1 on normal wafers: 2159 – 2846 (continuous)
original imputer mean  sensor_0: 2732  (true non-legacy 3014)
original imputer mean  sensor_1: 2281  (true non-legacy 2496)
original scaler std    sensor_1: 674  (true non-legacy 80)  -> genuine variation squashed
defect rate: legacy 0.0% vs normal 7.3%  -> the flag itself carries signal


## 2 · Repaired preprocessing (causal: fit on the past only)

- **Chronological split:** first 60 % of lots for train, next 20 % for validation (early stopping and threshold), last 20 % for test.
- **Legacy handling:** `is_legacy` flag plus one-hot `fw_code_100/200/300`. For legacy rows, `sensor_0` and `sensor_1` are set to *missing* before scaling, so they never enter the scaler's statistics. After scaling they are held at a neutral 0, and the flag tells the model *why*. No continuous value is ever fabricated over the codes.
- The **scaler and imputer** are fit on the training window only. Train-constant sensors are dropped. Scaled values are clipped to ±10 σ to survive tool drift.

In [7]:
n = len(df); i_tr, i_va = int(0.6 * n), int(0.8 * n)
splits = {"train": df.iloc[:i_tr], "val": df.iloc[i_tr:i_va], "test": df.iloc[i_va:]}
for k, d in splits.items():
    print(f"{k:5s} {len(d):4d} lots | {d.timestamp.min()} → {d.timestamp.max()} | defect rate {(d.label==1).mean():.2%}")
assert splits["train"].timestamp.max() <= splits["val"].timestamp.min() <= splits["test"].timestamp.min()

CODES = [100.0, 200.0, 300.0]

class WaferPreprocessor:
    """Fit on TRAIN only. transform() works on any single future lot."""
    def fit(self, d):
        Xc = self._continuous(d)
        self.keep_ = Xc.columns[Xc.nunique(dropna=True) > 1]          # FIX: drop train-constant channels
        self.scaler_ = StandardScaler().fit(Xc[self.keep_])           # NaN-aware: ignores masked/missing cells
        return self
    def _continuous(self, d):
        Xc = d[SENSOR_COLS].copy()
        legacy = d["sensor_0"].eq(0.0)
        Xc.loc[legacy, ["sensor_0", "sensor_1"]] = np.nan              # FIX: sentinel/codes are not measurements
        return Xc
    def transform(self, d):
        Xc = self.scaler_.transform(self._continuous(d)[self.keep_])
        Xc = np.nan_to_num(Xc, nan=0.0)                                # missing → train mean (0 in z-space)
        Xc = np.clip(Xc, -HP["clip"], HP["clip"])
        legacy = d["sensor_0"].eq(0.0).to_numpy()
        codes = np.stack([(legacy & d["sensor_1"].eq(c).to_numpy()) for c in CODES], 1)
        unknown_code = legacy & ~codes.any(1)
        if unknown_code.any(): print(f"WARNING: {unknown_code.sum()} legacy lots with unknown firmware code")
        cat = np.column_stack([legacy, codes]).astype(np.float64)      # FIX: categorical block, never scaled
        return np.hstack([Xc, cat]).astype(np.float32)
    @property
    def feature_names(self):
        return list(self.keep_) + ["is_legacy"] + [f"fw_code_{int(c)}" for c in CODES]

prep = WaferPreprocessor().fit(splits["train"])
X = {k: prep.transform(d) for k, d in splits.items()}
Y = {k: (d.label.to_numpy() == 1).astype(np.int64) for k, d in splits.items()}
F = X["train"].shape[1]
print(f"features: {F} ({len(prep.keep_)} continuous + 4 categorical) | shapes:", {k: v.shape for k, v in X.items()})
assert not any(np.isnan(v).any() for v in X.values())

train  940 lots | 2008-07-19 11:55:00 → 2008-09-20 05:34:00 | defect rate 8.09%
val    313 lots | 2008-09-20 06:08:00 → 2008-10-02 19:25:00 | defect rate 3.51%
test   314 lots | 2008-10-02 20:54:00 → 2008-10-17 06:07:00 | defect rate 5.41%


features: 472 (468 continuous + 4 categorical) | shapes: {'train': (940, 472), 'val': (313, 472), 'test': (314, 472)}


## 3 · Model: supervised autoencoder (input shape fixed)

Class imbalance is handled with `pos_weight = n_pass / n_fail` in the BCE, computed from the training window only. SMOTE-on-train-only is kept as an ablation in Section 6, so the choice is evidence-based.

In [8]:
class WaferSAE(nn.Module):
    def __init__(self, d_in, d_hid, d_lat):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hid), nn.LayerNorm(d_hid), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(d_hid, d_hid // 2), nn.LayerNorm(d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_lat))
        self.decoder = nn.Sequential(
            nn.Linear(d_lat, d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_hid), nn.GELU(),
            nn.Linear(d_hid, d_in))
        self.head = nn.Linear(d_lat, 1)
    def forward(self, x):
        assert x.dim() == 2, f"expected [B, F], got {tuple(x.shape)}"   # FIX: no silent channel axis
        z = self.encoder(x)
        return self.decoder(z), self.head(z).squeeze(-1), z

def make_loader(Xn, yn, shuffle, seed):
    ds = TensorDataset(torch.from_numpy(Xn).float(), torch.from_numpy(yn).float())   # FIX: [N, F], no unsqueeze
    g = torch.Generator().manual_seed(seed)
    return DataLoader(ds, batch_size=HP["batch_size"], shuffle=shuffle, generator=g if shuffle else None)

@torch.no_grad()
def predict(model, Xn):
    model.eval()
    _, logits, _ = model(torch.from_numpy(Xn).float().to(DEVICE))
    return torch.sigmoid(logits).cpu().numpy()

def train_model(Xtr, ytr, Xva, yva, seed, use_smote=False, verbose=False):
    set_seed(seed)
    if use_smote:   # ablation: oversample INSIDE the training window only
        Xtr, ytr = SMOTE(random_state=seed).fit_resample(Xtr, ytr)
        Xtr = Xtr.astype(np.float32); pos_w = 1.0
    else:
        pos_w = (ytr == 0).sum() / max((ytr == 1).sum(), 1)
    model = WaferSAE(Xtr.shape[1], HP["hidden"], HP["latent"]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_w, device=DEVICE))
    loader = make_loader(Xtr, ytr, True, seed)
    best, best_ap, bad, hist = None, -1, 0, []
    for epoch in range(1, HP["epochs"] + 1):
        model.train(); tot = rec_tot = 0.0
        for bx, by in loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            recon, logits, _ = model(bx)
            assert recon.shape == bx.shape, (recon.shape, bx.shape)       # FIX: shape guards
            assert logits.shape == by.shape, (logits.shape, by.shape)
            recon_loss = torch.mean((recon - bx) ** 2)
            loss = recon_loss + HP["lambda_clf"] * bce(logits, by)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += loss.item(); rec_tot += recon_loss.item()
        ap = average_precision_score(yva, predict(model, Xva))            # early stopping on VALIDATION
        hist.append((epoch, tot / len(loader), rec_tot / len(loader), ap))
        if verbose and (epoch == 1 or epoch % 5 == 0):
            print(f"epoch {epoch:02d} | loss {hist[-1][1]:.4f} | recon {hist[-1][2]:.4f} | val AP {ap:.4f}")
        if ap > best_ap: best_ap, best, bad = ap, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= HP["patience"]: break
    model.load_state_dict(best)
    return model, best_ap, hist

## 4 · Train (primary seed)

In [9]:
model, val_ap, hist = train_model(X["train"], Y["train"], X["val"], Y["val"], SEED, verbose=True)
print(f"best validation AP: {val_ap:.4f} (prevalence baseline {Y['val'].mean():.4f})")
print(f"final reconstruction MSE: {hist[-1][2]:.4f}  (original broadcast-bugged run plateaued at ~0.68)")

epoch 01 | loss 2.0397 | recon 0.7953 | val AP 0.0335


epoch 05 | loss 0.7636 | recon 0.7150 | val AP 0.0388


epoch 10 | loss 0.6203 | recon 0.6167 | val AP 0.0376


best validation AP: 0.0393 (prevalence baseline 0.0351)
final reconstruction MSE: 0.5665  (original broadcast-bugged run plateaued at ~0.68)


## 5 · Threshold from validation, single evaluation on the held-out future test window

**Metric choice.** At 6.6 % prevalence, accuracy and ROC-AUC look flattering. The primary metric is **average precision (PR-AUC)**, compared against its no-skill baseline (= prevalence), with F1, precision and recall reported at the operating threshold. The threshold maximises F1 on the *validation* window and is then frozen.

In [10]:
def pick_threshold(y, p):
    pr, rc, th = precision_recall_curve(y, p)
    f1 = 2 * pr * rc / np.clip(pr + rc, 1e-12, None)
    return float(th[np.nanargmax(f1[:-1])])

def report(y, p, thr, name):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    m = dict(F1=f1_score(y, pred, zero_division=0), precision=precision_score(y, pred, zero_division=0),
             recall=recall_score(y, pred, zero_division=0), ROC_AUC=roc_auc_score(y, p),
             AP=average_precision_score(y, p), prevalence=y.mean(), TN=tn, FP=fp, FN=fn, TP=tp)
    print(f"[{name}] prevalence {m['prevalence']:.2%} | thr {thr:.3f} | F1 {m['F1']:.4f} | P {m['precision']:.4f} | "
          f"R {m['recall']:.4f} | ROC-AUC {m['ROC_AUC']:.4f} | AP {m['AP']:.4f}")
    print(f"           confusion TN={tn} FP={fp} FN={fn} TP={tp}")
    return m

thr = pick_threshold(Y["val"], predict(model, X["val"]))
m_test = report(Y["test"], predict(model, X["test"]), thr, "TEST seed 1337")

[TEST seed 1337] prevalence 5.41% | thr 0.004 | F1 0.1290 | P 0.0748 | R 0.4706 | ROC-AUC 0.5787 | AP 0.0752
           confusion TN=198 FP=99 FN=9 TP=8


## 6 · Robustness: 5 seeds, SMOTE ablation, linear baseline

A single seed on 17 test defects is noisy, so the headline is reported as mean ± std over 5 seeds.

In [11]:
SEEDS = [1337, 1, 2, 3, 4]
rows = []
for variant, smote_flag in [("SAE + pos_weight", False), ("SAE + SMOTE(train only)", True)]:
    for s in SEEDS:
        mdl, _, _ = train_model(X["train"], Y["train"], X["val"], Y["val"], s, use_smote=smote_flag)
        t = pick_threshold(Y["val"], predict(mdl, X["val"]))
        p = predict(mdl, X["test"]); pred = (p >= t).astype(int)
        rows.append(dict(variant=variant, seed=s, F1=f1_score(Y["test"], pred, zero_division=0),
                         precision=precision_score(Y["test"], pred, zero_division=0),
                         recall=recall_score(Y["test"], pred, zero_division=0),
                         ROC_AUC=roc_auc_score(Y["test"], p), AP=average_precision_score(Y["test"], p)))
lr = LogisticRegression(C=0.01, class_weight="balanced", max_iter=5000).fit(X["train"], Y["train"])
t = pick_threshold(Y["val"], lr.predict_proba(X["val"])[:, 1]); p = lr.predict_proba(X["test"])[:, 1]; pred = (p >= t).astype(int)
rows.append(dict(variant="LogReg baseline", seed=0, F1=f1_score(Y["test"], pred, zero_division=0),
                 precision=precision_score(Y["test"], pred, zero_division=0), recall=recall_score(Y["test"], pred, zero_division=0),
                 ROC_AUC=roc_auc_score(Y["test"], p), AP=average_precision_score(Y["test"], p)))
res = pd.DataFrame(rows)
g = res.groupby("variant")[["F1", "precision", "recall", "ROC_AUC", "AP"]]
summary = g.mean().round(3).astype(str) + " ± " + g.std().fillna(0).round(3).astype(str)
print(f"test window prevalence: {Y['test'].mean():.2%} ({Y['test'].sum()} defects / {len(Y['test'])} lots) | AP no-skill baseline = {Y['test'].mean():.3f}")
print(summary.to_string())

test window prevalence: 5.41% (17 defects / 314 lots) | AP no-skill baseline = 0.054
                                    F1      precision         recall        ROC_AUC             AP
variant                                                                                           
LogReg baseline            0.163 ± 0.0    0.093 ± 0.0    0.647 ± 0.0    0.641 ± 0.0    0.091 ± 0.0
SAE + SMOTE(train only)  0.107 ± 0.061  0.062 ± 0.036  0.424 ± 0.289   0.59 ± 0.068  0.088 ± 0.021
SAE + pos_weight         0.117 ± 0.043  0.098 ± 0.062  0.353 ± 0.208  0.594 ± 0.096  0.097 ± 0.037


## 7 · Release gate (honest)

In [12]:
GATE_F1 = 0.90
print(f"Original notebook F1: 0.9915 on a 50/50, leaked test set -> invalid")
print(f"Repaired F1 at natural prevalence: {m_test['F1']:.4f} (seed {SEED}); "
      f"5-seed mean {res[res.variant=='SAE + pos_weight'].F1.mean():.4f}")
print("Release gate:", "PASSED" if m_test["F1"] >= GATE_F1 else "FAILED -> NO-GO")
os.makedirs("artifacts", exist_ok=True)
torch.save(model.state_dict(), "artifacts/wafer_sae_fixed.pt")
joblib.dump({"preprocessor": prep, "threshold": thr, "features": prep.feature_names}, "artifacts/preprocessor_fixed.joblib")
print("artifacts saved (for review only, not for deployment)")

Original notebook F1: 0.9915 on a 50/50, leaked test set -> invalid
Repaired F1 at natural prevalence: 0.1290 (seed 1337); 5-seed mean 0.1168
Release gate: FAILED -> NO-GO
artifacts saved (for review only, not for deployment)
